# cAPTure: packet and history MLP development screening

This GPU notebook trains two matched feed-forward baselines on the existing cAPTure development folds:

- `MLP-P` uses the same 103 fold-preprocessed packet features as XGB-P.
- `MLP-History` adds only the six causal features computed from the six complete preceding five-second windows.

Both variants reuse the frozen preprocessing, scenario/class weights, fold assignments, seed, and out-of-fold evaluation protocol. Training uses ten fixed epochs and never consults the outer validation scenarios for checkpoint selection. The primary operational decision time remains the five-second window close so the results are directly comparable with the XGB tables. Packet-arrival timing remains available for a later sensitivity audit. Held-out author-train and final-test scenarios are not read.


## 1. Prepare the Colab GPU environment

Select a GPU runtime before running this section. The notebook writes final models and OOF scores to Drive and uses `/content` only for temporary training matrices.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_WORK_ROOT = Path("/content/capture_mlp_work")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

required_files = [
    "code/python/requirements-capture-mlp.txt",
    "code/python/utils/capture_mlp.py",
    "code/python/utils/models.py",
    "configs/capture_experiment_v1.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
import torch
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA-enabled Colab runtime before training the MLPs.")
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("PyTorch version:", torch.__version__)
print("CUDA device:", torch.cuda.get_device_name(0))


Mounted at /content/drive
Repository commit: fb38b909ff0df93d2e885390e7445e9d0bfca51a
PyTorch version: 2.11.0+cu128
CUDA device: Tesla T4


## 2. Bind the completed development artifacts

Set `MLP_RUN_ID` only when resuming an existing MLP run. The four fold/variant directories are immutable and are verified before reuse. The history model requires the completed context run used by XGB-P+T.


In [2]:
from utils.capture_mlp import (
    run_capture_mlp_fold,
    run_capture_mlp_operational_evaluation,
    summarize_capture_mlp_oof,
    validate_capture_mlp_fold_run,
    validate_capture_mlp_operational_evaluation,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
PREPARED_RUN_DIR = (
    DRIVE_ROOT / "prepared_runs/20260917T235058_827743Z_prepare_full_dev"
)
PREPROCESSING_AUDIT_DIR = (
    DRIVE_ROOT / "preprocessing_runs/20260919T004143_161396Z_preprocessing"
)
CONTEXT_RUN_ID = "20260919T201909_754628Z_xgb_p_t"
CONTEXT_DIR = DRIVE_ROOT / "xgb_p_t_context_runs" / CONTEXT_RUN_ID

MLP_RUN_ID = None  # Set an existing ID only when resuming.
if MLP_RUN_ID is None:
    MLP_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_mlp_screen"
    )
MLP_RUN_DIR = DRIVE_ROOT / "mlp_runs" / MLP_RUN_ID
MLP_OPERATIONAL_DIR = (
    DRIVE_ROOT / "mlp_operational_runs" / f"{MLP_RUN_ID}_operational"
)

for required_path in (
    MANIFEST_PATH,
    PACKET_SCHEMA_PATH,
    PREPROCESSING_SCHEMA_PATH,
    PREPARED_RUN_DIR,
    PREPROCESSING_AUDIT_DIR,
    CONTEXT_DIR,
):
    if not required_path.exists():
        raise FileNotFoundError(f"Required input is missing: {required_path}")

print("MLP run ID:", MLP_RUN_ID)
print("MLP output:", MLP_RUN_DIR)
print("Operational output:", MLP_OPERATIONAL_DIR)


MLP run ID: 20260922T194018_897295Z_mlp_screen
MLP output: /content/drive/MyDrive/capture_gate0/mlp_runs/20260922T194018_897295Z_mlp_screen
Operational output: /content/drive/MyDrive/capture_gate0/mlp_operational_runs/20260922T194018_897295Z_mlp_screen_operational


## 3. Define resumable fold execution

Each call either validates a completed immutable fold or trains it from scratch. Training materializes the fold matrix on local Colab storage, copies only the checkpoint, reports, scaler, and OOF predictions to Drive, and removes the temporary matrix afterward.


In [3]:
def train_or_verify(variant_name, fold):
    output_dir = MLP_RUN_DIR / variant_name / f"fold_{fold}"
    if output_dir.exists():
        print(f"Validating existing {variant_name} fold {fold}...")
        report = validate_capture_mlp_fold_run(output_dir, fold, variant_name)
    else:
        report = run_capture_mlp_fold(
            manifest_path=MANIFEST_PATH,
            packet_schema_path=PACKET_SCHEMA_PATH,
            preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH,
            prepared_run_dir=PREPARED_RUN_DIR,
            preprocessing_audit_dir=PREPROCESSING_AUDIT_DIR,
            context_dir=CONTEXT_DIR,
            output_dir=output_dir,
            local_work_root=LOCAL_WORK_ROOT,
            fold=fold,
            variant_name=variant_name,
        )
    gc.collect()
    torch.cuda.empty_cache()
    print(
        f"{variant_name} fold {fold}: "
        f"macro ROC-AUC={report['fold_macro_packet_roc_auc']:.6f}"
    )
    return report


## 4. Train MLP-P

This is the direct neural counterpart of XGB-P. It operates independently on each packet and receives no window or history summaries.


In [4]:
packet_fold_a = train_or_verify("packet", "A")


Materializing packet fold A with 103 features...
Training packet fold A on 2,675,496 packets...
Epoch 1/10: weighted loss=0.234879
Epoch 2/10: weighted loss=0.175187
Epoch 3/10: weighted loss=0.153524
Epoch 4/10: weighted loss=0.159186
Epoch 5/10: weighted loss=0.152528
Epoch 6/10: weighted loss=0.160606
Epoch 7/10: weighted loss=0.155438
Epoch 8/10: weighted loss=0.159308
Epoch 9/10: weighted loss=0.154364
Epoch 10/10: weighted loss=0.147952
Scoring held-out scenario train_dollar_char...
Scoring held-out scenario train_slash_char...
Scoring held-out scenario train_sub_exf...
Saved packet fold A to /content/drive/MyDrive/capture_gate0/mlp_runs/20260922T194018_897295Z_mlp_screen/packet/fold_A.
packet fold A: macro ROC-AUC=0.960778


In [5]:
packet_fold_b = train_or_verify("packet", "B")
packet_summary = summarize_capture_mlp_oof(MLP_RUN_DIR, "packet")
display(pd.DataFrame.from_dict(packet_summary["scenario_metrics"], orient="index"))
print(
    "MLP-P hierarchical macro OOF packet ROC-AUC:",
    packet_summary["hierarchical_macro_oof_packet_roc_auc"],
)


Materializing packet fold B with 103 features...
Training packet fold B on 11,533,878 packets...
Epoch 1/10: weighted loss=0.128939
Epoch 2/10: weighted loss=0.092211
Epoch 3/10: weighted loss=0.089030
Epoch 4/10: weighted loss=0.090454
Epoch 5/10: weighted loss=0.086712
Epoch 6/10: weighted loss=0.089239
Epoch 7/10: weighted loss=0.089107
Epoch 8/10: weighted loss=0.092092
Epoch 9/10: weighted loss=0.089932
Epoch 10/10: weighted loss=0.091301
Scoring held-out scenario train_empty_conn...
Scoring held-out scenario train_qos_mid...
Saved packet fold B to /content/drive/MyDrive/capture_gate0/mlp_runs/20260922T194018_897295Z_mlp_screen/packet/fold_B.
packet fold B: macro ROC-AUC=0.979343


,fold,packets,packet_roc_auc,packet_pr_auc_diagnostic
train_dollar_char,A,2882555,0.900892,0.920344
train_slash_char,A,6425516,0.993960,0.997803
train_sub_exf,A,2225807,0.987482,0.967122
train_empty_conn,B,1175779,0.989449,0.981016
train_qos_mid,B,1499717,0.969237,0.973396


MLP-P hierarchical macro OOF packet ROC-AUC: 0.9700605259240561


## 5. Train MLP-History

This model has the same architecture and training policy as MLP-P. Its only additional inputs are the six summaries from the preceding 30 seconds; the current five-second window is excluded.


In [6]:
history_fold_a = train_or_verify("history", "A")


Materializing history fold A with 109 features...
Training history fold A on 2,675,496 packets...
Epoch 1/10: weighted loss=0.155237
Epoch 2/10: weighted loss=0.054320
Epoch 3/10: weighted loss=0.038467
Epoch 4/10: weighted loss=0.033142
Epoch 5/10: weighted loss=0.031547
Epoch 6/10: weighted loss=0.028152
Epoch 7/10: weighted loss=0.026245
Epoch 8/10: weighted loss=0.025166
Epoch 9/10: weighted loss=0.036115
Epoch 10/10: weighted loss=0.027097
Scoring held-out scenario train_dollar_char...
Scoring held-out scenario train_slash_char...
Scoring held-out scenario train_sub_exf...
Saved history fold A to /content/drive/MyDrive/capture_gate0/mlp_runs/20260922T194018_897295Z_mlp_screen/history/fold_A.
history fold A: macro ROC-AUC=0.953104


In [7]:
history_fold_b = train_or_verify("history", "B")
history_summary = summarize_capture_mlp_oof(MLP_RUN_DIR, "history")
display(pd.DataFrame.from_dict(history_summary["scenario_metrics"], orient="index"))
print(
    "MLP-History hierarchical macro OOF packet ROC-AUC:",
    history_summary["hierarchical_macro_oof_packet_roc_auc"],
)


Materializing history fold B with 109 features...
Training history fold B on 11,533,878 packets...
Epoch 1/10: weighted loss=0.057702
Epoch 2/10: weighted loss=0.026957
Epoch 3/10: weighted loss=0.022508
Epoch 4/10: weighted loss=0.021232
Epoch 5/10: weighted loss=0.017550
Epoch 6/10: weighted loss=0.017556
Epoch 7/10: weighted loss=0.012964
Epoch 8/10: weighted loss=0.016668
Epoch 9/10: weighted loss=0.013838
Epoch 10/10: weighted loss=0.013098
Scoring held-out scenario train_empty_conn...
Scoring held-out scenario train_qos_mid...
Saved history fold B to /content/drive/MyDrive/capture_gate0/mlp_runs/20260922T194018_897295Z_mlp_screen/history/fold_B.
history fold B: macro ROC-AUC=0.993397


,fold,packets,packet_roc_auc,packet_pr_auc_diagnostic
train_dollar_char,A,2882555,0.914545,0.877445
train_slash_char,A,6425516,0.989823,0.996374
train_sub_exf,A,2225807,0.954944,0.880450
train_empty_conn,B,1175779,0.999221,0.998640
train_qos_mid,B,1499717,0.987574,0.991400


MLP-History hierarchical macro OOF packet ROC-AUC: 0.9732507833573627


## 6. Compare threshold-free packet ranking

The hierarchical macro first averages scenarios within each fold and then averages the two folds. The delta isolates the effect of the six history features for this fixed MLP configuration.


In [8]:
mlp_summaries = {
    "mlp_p": packet_summary,
    "mlp_history": history_summary,
}
ranking_rows = []
for model_name, summary in mlp_summaries.items():
    ranking_rows.append({
        "model": model_name,
        "hierarchical_macro_oof_packet_roc_auc": (
            summary["hierarchical_macro_oof_packet_roc_auc"]
        ),
        "hierarchical_macro_oof_packet_average_precision": (
            summary["hierarchical_macro_oof_packet_pr_auc_diagnostic"]
        ),
    })
ranking_table = pd.DataFrame(ranking_rows).set_index("model")
display(ranking_table)
print(
    "History minus packet ROC-AUC:",
    ranking_table.loc["mlp_history", "hierarchical_macro_oof_packet_roc_auc"]
    - ranking_table.loc["mlp_p", "hierarchical_macro_oof_packet_roc_auc"],
)


,hierarchical_macro_oof_packet_roc_auc,hierarchical_macro_oof_packet_average_precision
model,,
mlp_p,0.970061,0.969481
mlp_history,0.973251,0.956555


History minus packet ROC-AUC: 0.003190257433306587


## 7. Select operational thresholds and evaluate OOF alerts

This section applies the existing frozen budgets: one false-alert window per hour, one per 12 hours, and one per five minutes. Threshold selection uses only development OOF predictions. Alert availability is aligned to the five-second window close for direct comparison with XGB.


In [9]:
if MLP_OPERATIONAL_DIR.exists():
    operational_report = validate_capture_mlp_operational_evaluation(
        manifest_path=MANIFEST_PATH,
        mlp_run_dir=MLP_RUN_DIR,
        output_dir=MLP_OPERATIONAL_DIR,
    )
else:
    operational_report = run_capture_mlp_operational_evaluation(
        manifest_path=MANIFEST_PATH,
        mlp_run_dir=MLP_RUN_DIR,
        output_dir=MLP_OPERATIONAL_DIR,
    )

operational_rows = []
for model_name, model_report in operational_report["models"].items():
    for budget_name in operational_report["budget_order"]:
        threshold_report = model_report["thresholds"][budget_name]
        metrics = model_report["budgets"][budget_name]["hierarchical_macro"]
        operational_rows.append({
            "model": model_name,
            "budget": budget_name,
            "threshold": threshold_report["threshold"],
            "worst_fold_false_alert_windows_per_hour": (
                threshold_report["worst_fold_false_alert_windows_per_hour"]
            ),
            **metrics,
        })
operational_table = pd.DataFrame(operational_rows).set_index(["model", "budget"])
display(operational_table)


threshold  \
model       budget                         
mlp_p       one_per_hour        0.999916   
            one_per_12_hours    0.999916   
            one_per_5_minutes   0.996442   
mlp_history one_per_hour        0.999647   
            one_per_12_hours    0.999925   
            one_per_5_minutes   0.999622   

                               worst_fold_false_alert_windows_per_hour  \
model       budget                                                       
mlp_p       one_per_hour                                      0.000000   
            one_per_12_hours                                  0.000000   
            one_per_5_minutes                                 4.799245   
mlp_history one_per_hour                                      0.948023   
            one_per_12_hours                                  0.077236   
            one_per_5_minutes                                11.918435   

                               false_alert_windows_per_hour  packet_recall  \
model       budget                                                           
mlp_p       one_per_hour                           0.000000       0.307551   
            one_per_12_hours                       0.000000       0.307551   
            one_per_5_minutes                      4.102017       0.687885   
mlp_history one_per_hour                           0.811114       0.676290   
            one_per_12_hours                       0.038618       0.358752   
            one_per_5_minutes                      6.296321       0.683135   

                               packet_false_positive_rate  \
model       budget                                          
mlp_p       one_per_hour                     2.878927e-05   
            one_per_12_hours                 2.878927e-05   
            one_per_5_minutes                2.570930e-04   
mlp_history one_per_hour                     4.495481e-05   
            one_per_12_hours                 9.627725e-07   
            one_per_5_minutes                6.025563e-04   

                               sequence_detection_rate  \
model       budget                                       
mlp_p       one_per_hour                      0.171804   
            one_per_12_hours                  0.171804   
            one_per_5_minutes                 0.954924   
mlp_history one_per_hour                      0.803205   
            one_per_12_hours                  0.396740   
            one_per_5_minutes                 0.810214   

                               mean_miss_capped_latency_seconds  
model       budget                                               
mlp_p       one_per_hour                              74.107867  
            one_per_12_hours                          74.107867  
            one_per_5_minutes                          2.757267  
mlp_history one_per_hour                              16.049786  
            one_per_12_hours                          72.972729  
            one_per_5_minutes                         15.279579

## 8. Report classical packet-classification metrics

These metrics are computed over packets at the selected one-per-hour operational threshold. ROC-AUC and average precision remain threshold-free. Scenario rows expose heterogeneity; the summary uses the same fold-then-fold hierarchical macro as the rest of the study.


In [11]:
def safe_ratio(numerator, denominator):
    return numerator / denominator if denominator else 0.0


def f_beta(precision, recall, beta):
    beta_squared = beta ** 2
    denominator = beta_squared * precision + recall
    return (
        (1 + beta_squared) * precision * recall / denominator
        if denominator else 0.0
    )


classic_rows = []
primary_budget = "one_per_hour"
for model_name, model_report in operational_report["models"].items():
    variant_name = model_report["variant_name"]
    ranking_by_scenario = mlp_summaries[model_name]["scenario_metrics"]
    scenario_metrics = model_report["budgets"][primary_budget]["scenario_metrics"]
    for scenario, metrics in scenario_metrics.items():
        true_positives = metrics["packet_true_positives"]
        false_positives = metrics["packet_false_positives"]
        true_negatives = metrics["packet_true_negatives"]
        false_negatives = metrics["packet_false_negatives"]
        precision = safe_ratio(true_positives, true_positives + false_positives)
        recall = safe_ratio(true_positives, true_positives + false_negatives)
        classic_rows.append({
            "model": model_name,
            "scenario": scenario,
            "fold": metrics["fold"],
            "budget": primary_budget,
            "threshold": metrics["threshold"],
            "packet_precision": precision,
            "packet_recall": recall,
            "packet_f1": f_beta(precision, recall, 1),
            "packet_f2": f_beta(precision, recall, 2),
            "packet_roc_auc": ranking_by_scenario[scenario]["packet_roc_auc"],
            "packet_average_precision": ranking_by_scenario[scenario][
                "packet_pr_auc_diagnostic"
            ],
            "packet_false_positive_rate": safe_ratio(
                false_positives, false_positives + true_negatives
            ),
            "false_alert_windows_per_hour": metrics[
                "false_alert_windows_per_hour"
            ],
            "packet_true_positives": true_positives,
            "packet_false_positives": false_positives,
            "packet_true_negatives": true_negatives,
            "packet_false_negatives": false_negatives,
        })

classic_table = pd.DataFrame(classic_rows)
display(classic_table.set_index(["model", "scenario"]).sort_index())

summary_fields = [
    "packet_precision",
    "packet_recall",
    "packet_f1",
    "packet_f2",
    "packet_roc_auc",
    "packet_average_precision",
    "packet_false_positive_rate",
    "false_alert_windows_per_hour",
]
fold_means = (
    classic_table.groupby(["model", "fold"], as_index=False)[summary_fields].mean()
)
classic_hierarchical = (
    fold_means.groupby("model", as_index=True)[summary_fields].mean()
)
thresholds = {
    model_name: model_report["thresholds"][primary_budget]["threshold"]
    for model_name, model_report in operational_report["models"].items()
}
classic_hierarchical.insert(
    0,
    "threshold",
    [thresholds[model_name] for model_name in classic_hierarchical.index],
)
display(classic_hierarchical)


fold        budget  threshold  packet_precision  \
model       scenario                                                            
mlp_history train_dollar_char    A  one_per_hour   0.999647          0.999778   
            train_empty_conn     B  one_per_hour   0.999647          0.999954   
            train_qos_mid        B  one_per_hour   0.999647          0.999971   
            train_slash_char     A  one_per_hour   0.999647          0.999985   
            train_sub_exf        A  one_per_hour   0.999647          0.999473   
mlp_p       train_dollar_char    A  one_per_hour   0.999916          0.000000   
            train_empty_conn     B  one_per_hour   0.999916          0.999844   
            train_qos_mid        B  one_per_hour   0.999916          0.999903   
            train_slash_char     A  one_per_hour   0.999916          0.000000   
            train_sub_exf        A  one_per_hour   0.999916          0.000000   

                               packet_recall  packet_f1  packet_f2  \
model       scenario                                                 
mlp_history train_dollar_char       0.481949   0.650379   0.537643   
            train_empty_conn        0.662853   0.797233   0.710776   
            train_qos_mid           0.694733   0.819863   0.739904   
            train_slash_char        0.916332   0.956332   0.931923   
            train_sub_exf           0.623081   0.767620   0.673833   
mlp_p       train_dollar_char       0.000000   0.000000   0.000000   
            train_empty_conn        0.641511   0.781563   0.691044   
            train_qos_mid           0.588691   0.741075   0.641451   
            train_slash_char        0.000000   0.000000   0.000000   
            train_sub_exf           0.000000   0.000000   0.000000   

                               packet_roc_auc  packet_average_precision  \
model       scenario                                                      
mlp_history train_dollar_char        0.914545                  0.877445   
            train_empty_conn         0.999221                  0.998640   
            train_qos_mid            0.987574                  0.991400   
            train_slash_char         0.989823                  0.996374   
            train_sub_exf            0.954944                  0.880450   
mlp_p       train_dollar_char        0.900892                  0.920344   
            train_empty_conn         0.989449                  0.981016   
            train_qos_mid            0.969237                  0.973396   
            train_slash_char         0.993960                  0.997803   
            train_sub_exf            0.987482                  0.967122   

                               packet_false_positive_rate  \
model       scenario                                        
mlp_history train_dollar_char                    0.000074   
            train_empty_conn                     0.000017   
            train_qos_mid                        0.000020   
            train_slash_char                     0.000039   
            train_sub_exf                        0.000100   
mlp_p       train_dollar_char                    0.000000   
            train_empty_conn                     0.000058   
            train_qos_mid                        0.000058   
            train_slash_char                     0.000000   
            train_sub_exf                        0.000000   

                               false_alert_windows_per_hour  \
model       scenario                                          
mlp_history train_dollar_char                      1.114743   
            train_empty_conn                       0.530973   
            train_qos_mid                          0.817439   
            train_slash_char                       0.542986   
            train_sub_exf                          1.186339   
mlp_p       train_dollar_char                      0.000000   
            train_empty_conn                       0.000000   
            tra

,threshold,packet_precision,packet_recall,packet_f1,packet_f2,packet_roc_auc,packet_average_precision,packet_false_positive_rate,false_alert_windows_per_hour
model,,,,,,,,,
mlp_history,0.999647,0.999854,0.676290,0.799996,0.719903,0.973251,0.956555,0.000045,0.811114
mlp_p,0.999916,0.499937,0.307551,0.380660,0.333124,0.970061,0.969481,0.000029,0.000000


## 9. Interpretation boundary

This is a one-seed development screen of one frozen MLP configuration. It can answer whether a feed-forward nonlinear model benefits from the causal history fields under the current folds. It does not estimate seed variability, complete a neural hyperparameter search, or provide final generalization evidence. Keep Test1 and Test2 untouched until the model and graph protocol are frozen.
